In [ ]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.5.1+cu121
0.20.1+cu121


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.5.1+cu121
True


In [ ]:
!pip install evaluate

In [ ]:
!pip install --upgrade transformers peft
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

In [ ]:
import evaluate

In [ ]:
# 1. Load dataset
dataset = load_dataset("stanfordnlp/imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# Tokenization function
def tokenize(batch):
  return tokenizer(batch["text"], truncation=True)

tokenized_ds = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Remove text column
tokenized_ds = tokenized_ds.remove_columns("text")
tokenized_ds = tokenized_ds.rename_column("label", "labels")
tokenized_ds.set_format("torch")

In [ ]:
# Data collator (dynamic padding)
data_collator = DataCollatorWithPadding(tokenizer)

In [ ]:
# Model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Metric
accuracy = evaluate.load("accuracy")

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = logits.argmax(axis=-1)
  return accuracy.compute(predictions=preds, references=labels)

In [ ]:
# 8️⃣ Training args
args = TrainingArguments(
    output_dir="./sentiment",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.205215,0.180917,0.933120
2,0.121799,0.234115,0.940040


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3126, training_loss=0.18918421935058907, metrics={'train_runtime': 7333.9874, 'train_samples_per_second': 6.818, 'train_steps_per_second': 0.426, 'total_flos': 1.302353679597312e+16, 'train_loss': 0.18918421935058907, 'epoch': 2.0})

In [ ]:
# Save the trained model
model.save_pretrained("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")
print("Model saved to ./sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./sentiment_model


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move your model to the device
model.to(device)



BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
# Source - https://stackoverflow.com/a/74788189
# Posted by ricksanchezdev
# Retrieved 2026-06-20, License - CC BY-SA 4.0

model.to('cuda')


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

# Check if model exists
model_path = "./sentiment_model"

if os.path.exists(model_path):
    print("Loading saved model...")
    # Load the saved model and tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    print("Model loaded successfully!")
else:
    print("Model not found! Please train and save the model first.")
    # You could add training code here if needed
    exit()

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model on device: {device}")

# 🔟 Inference
text = "This movie was absolutely amazing!"
inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model(**inputs)
prediction = outputs.logits.argmax(dim=-1).item()
print(f"Prediction: {prediction} (0 = negative, 1 = positive)")

# Test with negative example
text2 = "This movie was terrible and boring!"
inputs2 = tokenizer(text2, return_tensors="pt").to(device)
outputs2 = model(**inputs2)
prediction2 = outputs2.logits.argmax(dim=-1).item()
print(f"Prediction: {prediction2} (0 = negative, 1 = positive)")

# Test with another example
text3 = "The acting was decent but the plot was predictable."
inputs3 = tokenizer(text3, return_tensors="pt").to(device)
outputs3 = model(**inputs3)
prediction3 = outputs3.logits.argmax(dim=-1).item()
print(f"Prediction: {prediction3} (0 = negative, 1 = positive)")

Loading saved model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!
Model on device: cuda
Prediction: 1 (0 = negative, 1 = positive)
Prediction: 0 (0 = negative, 1 = positive)
Prediction: 0 (0 = negative, 1 = positive)


In [ ]:
!pwd

/content
